## 1. Cargar el programa

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Simulador educativo de un sistema experto sobre conexión Wi-Fi.

Ejecutar con: python Simulador_sistema_experto.py
Requiere Python 3.8 o posterior. Solo usa la biblioteca estándar.
Las reglas son ejemplos didácticos; el programa usa las respuestas del usuario
y no inspecciona la red ni cambia la configuración de ningún dispositivo.
"""

from dataclasses import dataclass


# 1. BASE DE CONOCIMIENTOS: condiciones y conclusiones explícitas.
@dataclass(frozen=True)
class Regla:
    codigo: str
    condiciones: dict
    conclusion: tuple
    explicacion: str


REGLAS = (
    Regla("R1", {"router_encendido": "no"},
          ("diagnostico", "energia"),
          "SI el router está apagado, ENTONCES revisar su alimentación eléctrica."),
    Regla("R2", {"router_encendido": "si", "conectado_wifi": "no"},
          ("revisar", "wifi"),
          "SI el router está encendido y el equipo no está conectado, "
          "ENTONCES revisar la conexión Wi-Fi."),
    Regla("R3", {"revisar": "wifi", "wifi_activado": "no"},
          ("diagnostico", "wifi_apagado"),
          "SI se revisa Wi-Fi y está desactivado, ENTONCES activarlo en el equipo."),
    Regla("R4", {"revisar": "wifi", "wifi_activado": "si", "red_visible": "no"},
          ("diagnostico", "red_no_visible"),
          "SI Wi-Fi está activado y la red no aparece, "
          "ENTONCES revisar la cobertura y la emisión de la red."),
    Regla("R5", {"revisar": "wifi", "wifi_activado": "si", "red_visible": "si"},
          ("diagnostico", "acceso_wifi"),
          "SI la red aparece pero el equipo no se conecta, "
          "ENTONCES revisar el acceso a esa red."),
    Regla("R6", {"router_encendido": "si", "conectado_wifi": "si"},
          ("revisar", "internet"),
          "SI el router está encendido y el equipo está conectado, "
          "ENTONCES revisar la navegación por internet."),
    Regla("R7", {"revisar": "internet", "navega": "si"},
          ("diagnostico", "sin_falla"),
          "SI el equipo está conectado y puede navegar, "
          "ENTONCES no se observa una falla con estos datos."),
    Regla("R8", {"revisar": "internet", "navega": "no", "otros_navegan": "si"},
          ("diagnostico", "equipo"),
          "SI este equipo no navega pero otros en la misma red sí, "
          "ENTONCES revisar el equipo afectado."),
    Regla("R9", {"revisar": "internet", "navega": "no", "otros_navegan": "no"},
          ("diagnostico", "red_general"),
          "SI este equipo y los demás en la misma red no navegan, "
          "ENTONCES revisar una posible falla compartida de la red."),
)

# Cada pregunta incluye los hechos que deben cumplirse para hacerla.
PREGUNTAS = (
    ("router_encendido", "¿El router está encendido?", {}),
    ("conectado_wifi", "¿Tu equipo indica que está conectado a tu red Wi-Fi?",
     {"router_encendido": "si"}),
    ("wifi_activado", "¿El Wi-Fi está activado en tu equipo?",
     {"revisar": "wifi"}),
    ("red_visible", "¿El nombre de tu red aparece entre las redes disponibles?",
     {"revisar": "wifi", "wifi_activado": "si"}),
    ("navega", "¿Puedes abrir varias páginas distintas desde este equipo?",
     {"revisar": "internet"}),
    ("otros_navegan", "¿Otros equipos pueden navegar con esa MISMA red Wi-Fi "
     "y sin usar datos móviles?", {"revisar": "internet", "navega": "no"}),
)

DIAGNOSTICOS = {
    "energia": ("El router está apagado según tus respuestas.", (
        "Comprueba si hay energía en el tomacorriente y si el adaptador está conectado.",
        "Enciende el router y espera a que termine de iniciar.",
        "Repite la consulta para comprobar si existe otro problema.")),
    "wifi_apagado": ("El Wi-Fi del equipo está desactivado.", (
        "Activa el Wi-Fi del equipo y revisa el modo avión.",
        "Conéctate a tu red y vuelve a comprobar la navegación.")),
    "red_no_visible": ("El equipo no detecta el nombre de la red.", (
        "Acércate al router y actualiza la lista de redes disponibles.",
        "Comprueba con el responsable de la red si el Wi-Fi del router está habilitado.",
        "También puede haber una red oculta o una banda no compatible con el equipo.")),
    "acceso_wifi": ("La red aparece, pero el equipo no está conectado.", (
        "Selecciona la red correcta y verifica la contraseña con su responsable.",
        "Revisa el mensaje de error que aparece al intentar conectarte.",
        "Estas respuestas no permiten confirmar que la contraseña sea la causa.")),
    "sin_falla": ("No se observa una falla de navegación con los datos indicados.", (
        "Si el problema ocurre a veces, repite la consulta durante la falla.",)),
    "equipo": ("Posible problema limitado al equipo o a su acceso a la red.", (
        "Prueba otro navegador y vuelve a conectar el equipo a la red Wi-Fi.",
        "Si continúa, solicita revisar su configuración de red y sus permisos de acceso.")),
    "red_general": ("Posible problema compartido del router o del servicio.", (
        "Comprueba las luces de estado y los cables externos del router.",
        "Consulta si el proveedor reporta una interrupción del servicio.",
        "Las respuestas no distinguen entre una falla del router y una del proveedor.")),
}


# 2. MOTOR DE INFERENCIA: encadenamiento hacia adelante.
def cumple(condiciones, hechos):
    return all(hechos.get(clave) == valor for clave, valor in condiciones.items())


def inferir(respuestas):
    """Deriva hechos hasta que ninguna regla agregue información nueva.

    Devuelve una copia de los hechos y la secuencia de reglas aplicadas.
    'no_se' expresa falta de información: nunca se interpreta como 'no'.
    """
    hechos = dict(respuestas)
    traza = []
    while True:
        hubo_cambio = False
        for regla in REGLAS:
            if not cumple(regla.condiciones, hechos):
                continue
            clave, valor = regla.conclusion
            if clave in hechos:
                if hechos[clave] != valor:
                    raise ValueError("Hechos contradictorios al aplicar " + regla.codigo)
                continue
            hechos[clave] = valor
            traza.append(regla)
            hubo_cambio = True
        if not hubo_cambio:
            return hechos, traza


def siguiente_pregunta(respuestas, hechos):
    for clave, texto, condiciones in PREGUNTAS:
        if clave not in respuestas and cumple(condiciones, hechos):
            return clave, texto
    return None


# 3. MÓDULO DE EXPLICACIÓN: muestra hechos, conclusión y reglas usadas.
def mostrar_resultado(respuestas):
    hechos, traza = inferir(respuestas)
    print("\nRESULTADO ORIENTATIVO")
    diagnostico = hechos.get("diagnostico")
    if diagnostico:
        titulo, acciones = DIAGNOSTICOS[diagnostico]
        print(titulo)
        print("\nRecomendaciones:")
        for numero, accion in enumerate(acciones, 1):
            print(f"{numero}. {accion}")
    else:
        print("No hay información suficiente para emitir una recomendación específica.")
        print("Comprueba los datos que faltan y realiza una nueva consulta.")
    print("\nRespuestas utilizadas:")
    etiquetas = {"si": "Sí", "no": "No", "no_se": "No sé"}
    for clave, pregunta, _ in PREGUNTAS:
        if clave in respuestas:
            print(f"• {pregunta} {etiquetas.get(respuestas[clave], respuestas[clave])}")
    print("\nReglas aplicadas:")
    for regla in traza:
        print(f"• {regla.codigo}: {regla.explicacion}")
    if not traza:
        print("Ninguna regla pudo activarse con las respuestas disponibles.")
    return hechos


# 4. INTERFAZ: entrada de respuestas y menú de demostración.
def consultar():
    respuestas = {}  # Memoria de trabajo nueva en cada consulta.
    opciones = {"s": "si", "si": "si", "sí": "si", "1": "si",
                "n": "no", "no": "no", "0": "no",
                "?": "no_se", "no sé": "no_se", "no se": "no_se", "ns": "no_se"}
    print("\nConsulta sobre un equipo conectado por Wi-Fi a un router.")
    print("Responde s = sí, n = no, ? = no sé. Escribe salir para cancelar.")
    try:
        while True:
            hechos, _ = inferir(respuestas)
            if "diagnostico" in hechos:
                break
            pendiente = siguiente_pregunta(respuestas, hechos)
            if pendiente is None:
                break
            clave, pregunta = pendiente
            respuesta = input(pregunta + " [s/n/?]: ").strip().lower()
            if respuesta == "salir":
                print("Consulta cancelada.")
                return
            if respuesta not in opciones:
                print("Respuesta no válida. Escribe s, n, ? o salir.")
                continue
            respuestas[clave] = opciones[respuesta]
    except (EOFError, KeyboardInterrupt):
        print("\nConsulta cancelada.")
        return
    mostrar_resultado(respuestas)


def ejemplo():
    print("\nCASO SIMULADO: otros equipos navegan y este equipo no.")
    return mostrar_resultado({"router_encendido": "si", "conectado_wifi": "si",
                              "navega": "no", "otros_navegan": "si"})


def ver_reglas():
    print("\nBASE DE CONOCIMIENTOS: 9 reglas didácticas")
    for regla in REGLAS:
        print(f"{regla.codigo}. {regla.explicacion}")


def menu():
    print("SIMULADOR DE SISTEMA EXPERTO • CONEXIÓN A INTERNET")
    print("Ejemplo educativo basado en reglas; no realiza mediciones de la red.")
    acciones = {"1": consultar, "2": ejemplo, "3": ver_reglas}
    while True:
        print("\n1. Nueva consulta\n2. Ver un caso simulado\n3. Ver las reglas\n0. Salir")
        try:
            opcion = input("Elige una opción: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nSimulador finalizado.")
            return
        if opcion == "0":
            print("Simulador finalizado.")
            return
        if opcion in acciones:
            acciones[opcion]()
        else:
            print("Opción no válida. Elige 1, 2, 3 o 0.")


## 2. Hacer una consulta
Ejecuta la siguiente celda y responde a cada pregunta:

- `s`: sí.
- `n`: no.
- `?`: no sé.
- `salir`: cancela la consulta.

Se hacen hasta cuatro preguntas según tus respuestas. **No sé** se trata como información desconocida. Para realizar otra consulta, vuelve a ejecutar esta celda.


In [ ]:
consultar()


## 3. Ver un ejemplo ya resuelto
Caso: el router está encendido, el equipo está conectado al Wi-Fi, no logra navegar y otros equipos sí navegan en la misma red.

El motor aplica **R6 → R8**: primero deduce que corresponde revisar internet y luego recomienda revisar el equipo afectado. La salida siguiente se obtuvo ejecutando el programa.


In [5]:
resultado_ejemplo = ejemplo()



CASO SIMULADO: otros equipos navegan y este equipo no.

RESULTADO ORIENTATIVO
Posible problema limitado al equipo o a su acceso a la red.

Recomendaciones:
1. Prueba otro navegador y vuelve a conectar el equipo a la red Wi-Fi.
2. Si continúa, solicita revisar su configuración de red y sus permisos de acceso.

Respuestas utilizadas:
• ¿El router está encendido? Sí
• ¿Tu equipo indica que está conectado a tu red Wi-Fi? Sí
• ¿Puedes abrir varias páginas distintas desde este equipo? No
• ¿Otros equipos pueden navegar con esa MISMA red Wi-Fi y sin usar datos móviles? Sí

Reglas aplicadas:
• R6: SI el router está encendido y el equipo está conectado, ENTONCES revisar la navegación por internet.
• R8: SI este equipo no navega pero otros en la misma red sí, ENTONCES revisar el equipo afectado.


## 4. Consultar la base de reglas
Estas son las nueve reglas que sustentan las recomendaciones.


In [6]:
ver_reglas()



BASE DE CONOCIMIENTOS: 9 reglas didácticas
R1. SI el router está apagado, ENTONCES revisar su alimentación eléctrica.
R2. SI el router está encendido y el equipo no está conectado, ENTONCES revisar la conexión Wi-Fi.
R3. SI se revisa Wi-Fi y está desactivado, ENTONCES activarlo en el equipo.
R4. SI Wi-Fi está activado y la red no aparece, ENTONCES revisar la cobertura y la emisión de la red.
R5. SI la red aparece pero el equipo no se conecta, ENTONCES revisar el acceso a esa red.
R6. SI el router está encendido y el equipo está conectado, ENTONCES revisar la navegación por internet.
R7. SI el equipo está conectado y puede navegar, ENTONCES no se observa una falla con estos datos.
R8. SI este equipo no navega pero otros en la misma red sí, ENTONCES revisar el equipo afectado.
R9. SI este equipo y los demás en la misma red no navegan, ENTONCES revisar una posible falla compartida de la red.
